# Artificial Hivemind: where does the homogeneity come from?

This notebook replicates one result of "Artificial Hivemind: The Open-Ended Homogeneity of Language Models (and Beyond)" (Jiang et al., NeurIPS 2025) and extends it to the question the authors leave open: does the homogeneity come from pretraining or from post-training? Everything runs on Kaggle's free GPUs with open models: no API keys, no payment.

- Replication: Qwen2.5-1.5B-Instruct on the 100 Infinity-Chat eval prompts, 50 samples each (top-p 0.9, temperature 1.0), compared with the paper's Table 6.
- Extension: the same protocol on OLMo 2 1B at each training stage (base, SFT, DPO, final RLVR model).
- Analysis: base vs post-trained models in three families (OLMo 2, Qwen2.5, SmolLM2), within and across families.
- Ablations: response length (first 64 words only), a second embedding model, a lexical metric (trigram overlap), and an LLM judge that removes off-task responses.
- Idea: can decoding (higher temperature, min-p) or list prompting (5 answers per call) bring the final model back to base-model diversity?

How to run on Kaggle
- Settings: Accelerator "GPU T4 x2", Internet on (needs a phone-verified account, which is free).
- Save Version - "Save & Run All (Commit)" runs everything in the background, about 2 to 3 hours of the free weekly GPU quota. Results go to /kaggle/working/hivemind: figures, tables and every response.
- Re-running is safe: finished steps are skipped.

In [ ]:
# -- config --

import os, sys, re, json, glob, time, random, shutil, threading, subprocess, functools, warnings

import numpy as np

import pandas as pd

import requests

warnings.filterwarnings("ignore", category=RuntimeWarning)

WORK = os.environ.get("HIVEMIND_DIR", "/kaggle/working/hivemind")

VENV = os.environ.get("HIVEMIND_VENV", "/tmp/hivemind-venv")

VENV_PY = os.path.join(VENV, "bin", "python")

VLLM_VERSION = "0.25.1"

N_SAMPLES = 50

TOP_P = 0.9

TEMPERATURE = 1.0

CHAT_MAX_TOKENS = 2048

BASE_MAX_TOKENS = 512

TRUNC_WORDS = 64

JUDGE_PER_PROMPT = 10

RUN_MITIGATIONS = True

EMB_MAIN = "Qwen/Qwen3-Embedding-0.6B"

EMB_SECOND = "sentence-transformers/all-mpnet-base-v2"

JUDGE_MODELS = ["microsoft/Phi-3.5-mini-instruct", "Qwen/Qwen2.5-3B-Instruct"]

for sub in ["scripts", "gen", "logs", "emb", "judge", "figures", "tables"]:
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)

In [ ]:
# -- models and jobs --

def job(name, hf, kind, label, family, stage, n=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P, min_p=0.0, mode="standard"):
    max_tokens = BASE_MAX_TOKENS if kind == "base" else CHAT_MAX_TOKENS

    return dict(name=name, hf=hf, kind=kind, label=label, family=family, stage=stage, n=n, temperature=temperature, top_p=top_p, min_p=min_p, mode=mode, max_tokens=max_tokens)

JOBS = []

JOBS.append(job("olmo_base", "allenai/OLMo-2-0425-1B", "base", "OLMo base", "OLMo 2", "Base"))

JOBS.append(job("olmo_sft", "allenai/OLMo-2-0425-1B-SFT", "chat", "OLMo SFT", "OLMo 2", "SFT"))

JOBS.append(job("olmo_dpo", "allenai/OLMo-2-0425-1B-DPO", "chat", "OLMo DPO", "OLMo 2", "DPO"))

JOBS.append(job("olmo_final", "allenai/OLMo-2-0425-1B-Instruct", "chat", "OLMo final", "OLMo 2", "Final"))

JOBS.append(job("qwen_base", "Qwen/Qwen2.5-1.5B", "base", "Qwen base", "Qwen2.5", "Base"))

JOBS.append(job("qwen_inst", "Qwen/Qwen2.5-1.5B-Instruct", "chat", "Qwen inst", "Qwen2.5", "Final"))

JOBS.append(job("smol_base", "HuggingFaceTB/SmolLM2-1.7B", "base", "SmolLM base", "SmolLM2", "Base"))

JOBS.append(job("smol_inst", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "chat", "SmolLM inst", "SmolLM2", "Final"))

MAIN = [j["name"] for j in JOBS]

JOBS.append(job("olmo_final_t15", "allenai/OLMo-2-0425-1B-Instruct", "chat", "Final, T=1.5", "OLMo 2", "Final", n=20, temperature=1.5))

JOBS.append(job("olmo_final_minp", "allenai/OLMo-2-0425-1B-Instruct", "chat", "Final, min-p", "OLMo 2", "Final", n=20, temperature=2.0, top_p=1.0, min_p=0.1))

JOBS.append(job("olmo_final_list", "allenai/OLMo-2-0425-1B-Instruct", "chat", "Final, list x5", "OLMo 2", "Final", n=4, mode="list"))

MITIGATIONS = [j["name"] for j in JOBS if j["name"] not in MAIN]

JOB = {j["name"]: j for j in JOBS}

FAMILIES = ["OLMo 2", "Qwen2.5", "SmolLM2"]

STAGES = ["Base", "SFT", "DPO", "Final"]

## 1. Setup

vLLM is installed in its own virtual environment, so it cannot break Kaggle's preinstalled packages. All GPU work (generation, embeddings, judge) runs there as small scripts; the analysis runs in this notebook. vLLM 0.25.1 is pinned because it has been run on T4s. If it fails to install or to start, the scripts fall back to plain transformers (slower, same outputs).

In [ ]:
# -- setup: isolated vLLM environment --

def sh(cmd):
    print("$", cmd)

    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])

    return r.returncode

if not os.path.exists(VENV_PY):
    sh(f"{sys.executable} -m pip install -q uv")

    sh(f"{sys.executable} -m uv venv --python {sys.executable} {VENV}")

    if sh(f"{sys.executable} -m uv pip install --python {VENV_PY} vllm=={VLLM_VERSION}") != 0:
        sh(f"{sys.executable} -m uv pip install --python {VENV_PY} torch transformers accelerate")

VENV_ENV = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}

probe = subprocess.run([VENV_PY, "-c", "import vllm; print(vllm.__version__)"], capture_output=True, text=True, env=VENV_ENV)

BACKEND = "vllm" if probe.returncode == 0 else "hf"

print(subprocess.run([VENV_PY, "-c", "import torch, transformers; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)"], capture_output=True, text=True, env=VENV_ENV).stdout.strip())

gpu_list = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines()

N_GPUS = max(1, len(gpu_list))


def run_on_gpus(cmds, tag):
    codes = []

    for start in range(0, len(cmds), N_GPUS):
        procs = [subprocess.Popen(cmd, env=dict(VENV_ENV, CUDA_VISIBLE_DEVICES=str(g)), stdout=open(os.path.join(WORK, "logs", f"{tag}_{start + g}.log"), "a"), stderr=subprocess.STDOUT) for g, cmd in enumerate(cmds[start:start + N_GPUS])]

        codes += [p.wait() for p in procs]

    return codes

print("backend:", BACKEND, probe.stdout.strip(), "| GPUs:", gpu_list)

In [ ]:
# -- optional: a free Hugging Face token, only useful if model downloads get rate-limited --

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN = secret("HF_TOKEN")

if HF_TOKEN:
    VENV_ENV["HF_TOKEN"] = HF_TOKEN

print("HF token:", bool(HF_TOKEN))

In [ ]:
# -- prompts: the 100 Infinity-Chat eval queries used in the paper --

PROMPTS_PATH = os.path.join(WORK, "prompts.json")

if not os.path.exists(PROMPTS_PATH):
    r = requests.get("https://huggingface.co/datasets/liweijiang/infinite-chats-eval/resolve/main/data/train-00000-of-00001.parquet", timeout=120)

    r.raise_for_status()

    open("/tmp/infinite-chats-eval.parquet", "wb").write(r.content)

    json.dump(pd.read_parquet("/tmp/infinite-chats-eval.parquet")["query"].tolist(), open(PROMPTS_PATH, "w"))

PROMPTS = json.load(open(PROMPTS_PATH))

METAPHOR = next(i for i, q in enumerate(PROMPTS) if "metaphor" in q.lower())

print(len(PROMPTS), "prompts | metaphor prompt:", PROMPTS[METAPHOR])

## 2. Generation

Each model answers the 100 prompts 50 times with the paper's settings: top-p 0.9, temperature 1.0, up to 2048 new tokens. Chat models use their own chat template, including any default system prompt, as in the paper.

Base models have no chat template, so they get a plain completion prompt ("Below is a request from a user, followed by a response. Request: ... Response:"). They stop at the next "Request:" or at two blank lines, with a 512-token cap, since base models rarely end on their own.

Each job first runs a short greedy smoke test in float16 (T4s have no bfloat16). If the output is garbage, a sign of float16 overflow, the job reruns in float32. Two jobs run at a time, one per T4, and each model's download is deleted once no other job needs it.

In [ ]:
%%writefile {WORK}/scripts/gen.py
# -- generation script: runs inside the vLLM environment, one model per call --

import argparse, json, math, os, re, sys, time

STOP_BASE = ["\nRequest:", "Below is a request", "\n\n\n"]

BASE_TEMPLATE = "Below is a request from a user, followed by a response.\n\nRequest: {query}\n\nResponse:"

LIST_TEMPLATE = 'Give 5 different responses to the request below. Make them as different from each other as possible. Start each one on a new line with "Response 1:", "Response 2:" and so on.\n\nRequest: {query}'

HEAD_RE = re.compile(r"(?mi)^[ \t*#]*response[ \t]*[1-5][ \t]*[.):]*[* \t]*")

NUM_RE = re.compile(r"(?m)^[ \t*#]*[1-5][ \t]*[.)][* \t]*")


def build_ids(tok, query, kind, mode):
    if kind == "base":
        return tok(BASE_TEMPLATE.format(query=query)).input_ids

    content = LIST_TEMPLATE.format(query=query) if mode == "list" else query

    text = tok.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)

    return tok(text, add_special_tokens=False).input_ids


def cut_stops(text, stops):
    for s in stops:
        k = text.find(s)

        if k >= 0:
            text = text[:k]

    return text.strip()


def split_list(text):
    pattern = HEAD_RE if HEAD_RE.search(text) else NUM_RE

    marks = list(pattern.finditer(text))

    items = [text[a.end():b.start() if b else len(text)].strip().strip('"').strip() for a, b in zip(marks, marks[1:] + [None])]

    return [t for t in items if t][:5]


def degenerate(smoke, kind):
    bad = 0

    for text, logprob in smoke:
        s = text.strip()

        top = max(s.count(c) for c in set(s)) if s else 0

        nan = logprob is not None and not math.isfinite(logprob)

        if nan or (len(s) >= 20 and top >= 0.5 * len(s)) or (not s and kind == "chat"):
            bad += 1

    return bad > len(smoke) // 2


def run_vllm(args, prompts, dtype):
    from vllm import LLM, SamplingParams

    llm = LLM(model=args.model, dtype=dtype, seed=args.seed, gpu_memory_utilization=0.85, max_model_len=args.max_tokens + 512, max_num_seqs=256, enforce_eager=True)

    tok = llm.get_tokenizer()

    reqs = [{"prompt_token_ids": build_ids(tok, q, args.kind, args.mode)} for q in prompts]

    stops = STOP_BASE if args.kind == "base" else []

    smoke = llm.generate(reqs[:4], SamplingParams(temperature=0.0, max_tokens=48, stop=stops, logprobs=1), use_tqdm=False)

    smoke = [(cut_stops(o.outputs[0].text, stops), o.outputs[0].cumulative_logprob) for o in smoke]

    print("smoke test:", json.dumps([t for t, _ in smoke])[:1500], flush=True)

    if dtype != "float32" and degenerate(smoke, args.kind):
        return None

    sp = SamplingParams(n=args.n, temperature=args.temperature, top_p=args.top_p, min_p=args.min_p, max_tokens=args.max_tokens, stop=stops)

    outs = llm.generate(reqs, sp, use_tqdm=True)

    return [[(cut_stops(c.text, stops), len(c.token_ids), c.finish_reason) for c in o.outputs] for o in outs]


def run_hf(args, prompts, dtype):
    import torch

    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(args.model)

    model = AutoModelForCausalLM.from_pretrained(args.model).to(torch.float32 if dtype == "float32" else torch.float16).cuda().eval()

    ids = [build_ids(tok, q, args.kind, args.mode) for q in prompts]

    with torch.no_grad():
        finite = all(bool(torch.isfinite(model(torch.tensor([x]).cuda()).logits).all()) for x in ids[:4])

    if dtype != "float32" and not finite:
        return None

    stops = STOP_BASE if args.kind == "base" else []

    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

    torch.manual_seed(args.seed)

    res = []

    for i, x in enumerate(ids):
        row = []

        while len(row) < args.n:
            k = min(25 if dtype == "half" else 8, args.n - len(row))

            out = model.generate(torch.tensor([x]).cuda(), do_sample=True, temperature=args.temperature, top_p=args.top_p, min_p=args.min_p or None, max_new_tokens=args.max_tokens, num_return_sequences=k, pad_token_id=pad, stop_strings=stops or None, tokenizer=tok)

            for seq in out[:, len(x):]:
                n_tok = int((seq != pad).sum())

                row.append((cut_stops(tok.decode(seq, skip_special_tokens=True), stops), n_tok, "length" if n_tok >= args.max_tokens else "stop"))

        res.append(row)

        print(f"prompt {i + 1}/{len(ids)}", flush=True)

    return res


def main():
    p = argparse.ArgumentParser()

    for name, default in [("model", None), ("kind", "chat"), ("mode", "standard"), ("prompts", None), ("out", None), ("backend", "vllm"), ("dtype", "half")]:
        p.add_argument("--" + name, default=default)

    for name, default in [("n", 50), ("max_tokens", 2048), ("seed", 0)]:
        p.add_argument("--" + name, type=int, default=default)

    for name, default in [("temperature", 1.0), ("top_p", 0.9), ("min_p", 0.0)]:
        p.add_argument("--" + name, type=float, default=default)

    args = p.parse_args()

    prompts = json.load(open(args.prompts))

    t0 = time.time()

    res = (run_vllm if args.backend == "vllm" else run_hf)(args, prompts, args.dtype)

    if res is None:
        print("float16 output looks degenerate, exiting with code 3", flush=True)

        sys.exit(3)

    with open(args.out + ".tmp", "w") as f:
        for i, r in enumerate(res):
            row = dict(pid=i, responses=[t for t, _, _ in r], ntok=[n for _, n, _ in r], finish=[x for _, _, x in r], dtype=args.dtype, backend=args.backend)

            if args.mode == "list":
                row["raw"] = row["responses"]

                row["parsed"] = [len(split_list(t)) for t in row["raw"]]

                row["responses"] = [item for t in row["raw"] for item in split_list(t)]

            f.write(json.dumps(row) + "\n")

    os.replace(args.out + ".tmp", args.out)

    print(f"done in {(time.time() - t0) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# -- run all generation jobs, one per GPU at a time --

def gen_path(name):
    return os.path.join(WORK, "gen", f"{name}.jsonl")


def gen_cmd(j, backend, dtype):
    args = dict(model=j["hf"], kind=j["kind"], mode=j["mode"], prompts=PROMPTS_PATH, out=gen_path(j["name"]), backend=backend, dtype=dtype, n=j["n"], max_tokens=j["max_tokens"], temperature=j["temperature"], top_p=j["top_p"], min_p=j["min_p"])

    return [VENV_PY, os.path.join(WORK, "scripts", "gen.py")] + [x for k, v in args.items() for x in ("--" + k, str(v))]


def run_job(j, gpu):
    env = dict(VENV_ENV, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false")

    attempts = [(BACKEND, "half")]

    while attempts:
        backend, dtype = attempts.pop(0)

        with open(os.path.join(WORK, "logs", j["name"] + ".log"), "a") as log:
            code = subprocess.run(gen_cmd(j, backend, dtype), env=env, stdout=log, stderr=subprocess.STDOUT).returncode

        if code == 0:
            return f"ok ({backend}, {dtype})"

        if code == 3:
            attempts.insert(0, (backend, "float32"))
        elif backend == "vllm":
            attempts.append(("hf", dtype))

    return "FAILED"


def worker(gpu):
    while True:
        with LOCK:
            if not TODO:
                return

            j = TODO.pop(0)

        t0 = time.time()

        STATUS[j["name"]] = run_job(j, gpu)

        with LOCK:
            USES[j["hf"]] -= 1

            last = USES[j["hf"]] == 0

        if last:
            shutil.rmtree(os.path.join(HUB, "models--" + j["hf"].replace("/", "--")), ignore_errors=True)

        print(f"[GPU {gpu}] {j['name']}: {STATUS[j['name']]} in {(time.time() - t0) / 60:.1f} min", flush=True)

HUB = os.environ.get("HF_HUB_CACHE", os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "hub"))

TODO = [j for j in JOBS if (j["name"] in MAIN or RUN_MITIGATIONS) and not os.path.exists(gen_path(j["name"]))]

USES = {}

for j in TODO:
    USES[j["hf"]] = USES.get(j["hf"], 0) + 1

LOCK = threading.Lock()

STATUS = {}

threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPUS)]

for t in threads:
    t.start()

for t in threads:
    t.join()

for name, status in STATUS.items():
    if status == "FAILED":
        print(f"\n{name} failed, end of logs/{name}.log:\n" + "".join(open(os.path.join(WORK, "logs", name + ".log")).readlines()[-25:]))

print("generation finished:", len(STATUS), "jobs run,", sum(s == "FAILED" for s in STATUS.values()), "failed")

In [ ]:
# -- check the generations --

@functools.lru_cache(maxsize=None)
def load_gen(name):
    return [json.loads(line) for line in open(gen_path(name))]


def words(t):
    return len(t.split())

DONE = [j["name"] for j in JOBS if os.path.exists(gen_path(j["name"]))]

rows = []

for name in DONE:
    g = load_gen(name)

    resp = [t for r in g for t in r["responses"]]

    fin = [f for r in g for f in r["finish"]]

    rows.append(dict(job=name, prompts=len(g), responses=len(resp), words=np.mean([words(t) for t in resp]), empty_pct=100 * np.mean([not t.strip() for t in resp]), capped_pct=100 * np.mean([f == "length" for f in fin]), dtype=g[0]["dtype"], backend=g[0]["backend"]))

CHECK = pd.DataFrame(rows).set_index("job").round(1)

print(CHECK.to_string())

if "olmo_final_list" in DONE:
    print(f"\nlist prompting: {np.mean([k for r in load_gen('olmo_final_list') for k in r['parsed']]):.2f} of 5 answers parsed per call")

for name in [n for n in MAIN if n in DONE]:
    print(f"\n{JOB[name]['label']}:", *["  - " + t[:160].replace("\n", " ") for t in load_gen(name)[METAPHOR]["responses"][:3]], sep="\n")

## 3. Embeddings

Similarity follows the paper: cosine similarity of sentence embeddings, averaged over all pairs of responses to the same prompt. The paper used OpenAI's text-embedding-3-small, which is paid, so here the main model is Qwen3-Embedding-0.6B: open, strong on retrieval and similarity benchmarks, with a context long enough for full responses. Two controls:
- the first 64 words of each response only, because base and post-trained models write at different lengths, and length alone changes embedding similarity
- a second, unrelated embedding model (all-mpnet-base-v2, mean pooling)

The two models run in parallel, one per T4.

In [ ]:
%%writefile {WORK}/scripts/embed.py
# -- sentence embeddings with plain transformers: last-token pooling (Qwen3-Embedding) or mean pooling (mpnet) --

import json, sys

import numpy as np

import torch

from transformers import AutoModel, AutoTokenizer


def encode(model, tok, texts, pooling, max_len):
    order = sorted(range(len(texts)), key=lambda i: -len(texts[i]))

    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)

    start = 0

    with torch.no_grad():
        while start < len(order):
            idx = order[start:start + max(1, min(256, 24000 // (len(texts[order[start]]) // 3 + 16)))]

            batch = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to("cuda")

            hidden = model(**batch).last_hidden_state.float()

            mask = batch["attention_mask"].unsqueeze(-1).float()

            emb = hidden[:, -1] if pooling == "last" else (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)

            out[idx] = torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()

            start += len(idx)

    return out


def main():
    model_name, pooling, max_len, src, dst = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4], sys.argv[5]

    texts = json.load(open(src))

    tok = AutoTokenizer.from_pretrained(model_name, padding_side="left" if pooling == "last" else "right")

    for dtype in [torch.float16, torch.float32]:
        model = AutoModel.from_pretrained(model_name).to(dtype).cuda().eval()

        emb = encode(model, tok, texts, pooling, max_len)

        if np.isfinite(emb).all():
            break

        print("non-finite embeddings in float16, retrying in float32", flush=True)

    np.save(dst, emb.astype(np.float16))

    print("embedded", len(texts), "texts with", model_name, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# -- embeddings: Qwen3-Embedding (full responses and first 64 words) and mpnet (first 64 words) --

def trunc(t, k=TRUNC_WORDS):
    return " ".join(t.split()[:k])


def clean(t):
    return t.strip()[:20000] or "[empty]"


def flat(name):
    g = load_gen(name)

    return [t for r in g for t in r["responses"]], np.array([r["pid"] for r in g for _ in r["responses"]])


def emb_path(name, variant):
    return os.path.join(WORK, "emb", f"{name}_{variant}.npy")


def load_emb(name, variant):
    path = emb_path(name, variant)

    return np.load(path).astype(np.float32) if os.path.exists(path) else None

EMBEDDERS = {"main": (EMB_MAIN, "last", 2048), "second": (EMB_SECOND, "mean", 384)}

VARIANTS = {"main_full": ("main", False), "main_trunc": ("main", True), "second_trunc": ("second", True)}

PLAN = {e: [(n, v) for n in DONE for v, (ev, _) in VARIANTS.items() if ev == e and (v == "main_trunc" or n in MAIN) and not os.path.exists(emb_path(n, v))] for e in EMBEDDERS}

cmds, parts = [], []

for e, todo in PLAN.items():
    if todo:
        src = os.path.join(WORK, "emb", f"_{e}_in.json")

        json.dump([clean(trunc(t)) if VARIANTS[v][1] else clean(t) for n, v in todo for t in flat(n)[0]], open(src, "w"))

        cmds.append([VENV_PY, os.path.join(WORK, "scripts", "embed.py"), *map(str, EMBEDDERS[e]), src, src.replace(".json", ".npy")])

        parts.append((e, todo, src))

for (e, todo, src), code in zip(parts, run_on_gpus(cmds, "embed")):
    if code != 0:
        print(f"embedding with {EMBEDDERS[e][0]} failed, see logs/embed_*.log")

        continue

    E = np.load(src.replace(".json", ".npy"))

    start = 0

    for n, v in todo:
        k = len(flat(n)[0])

        np.save(emb_path(n, v), E[start:start + k])

        start += k

    os.remove(src)

    os.remove(src.replace(".json", ".npy"))

    print("embedded with", EMBEDDERS[e][0] + ":", ", ".join(f"{n}/{v}" for n, v in todo))

## 4. LLM judge

Base models can look diverse just by producing off-task text. A judge model grades the first 10 responses per prompt: 1 = not a real response to the request, 2 = a poor response, 3 = a reasonable response. Grades 2 and 3 count as valid, and similarity is then recomputed on valid responses only.

The judge runs locally with vLLM: Phi-3.5-mini-instruct, which comes from none of the three families studied, so it has no reason to favor one family's style. If it does not answer with a digit, Qwen2.5-3B-Instruct takes over. Responses are cut to 300 words.

In [ ]:
%%writefile {WORK}/scripts/judge.py
# -- local LLM judge: one digit per (request, response) pair --

import json, re, sys

JUDGE_PROMPT = "You are grading a response to a request.\n\nRequest: {q}\n\nResponse: {r}\n\nGrade the response with one digit:\n1 - not a real response to the request (off-topic, incoherent, repeats the request, or continues some unrelated text)\n2 - a response to the request, but a poor one (partly off-task, unfinished, or with major errors)\n3 - a reasonable response to the request\n\nAnswer with the digit only."


def make_vllm(model_name):
    from vllm import LLM, SamplingParams

    llm = LLM(model=model_name, dtype="half", seed=0, gpu_memory_utilization=0.85, max_model_len=2048, enforce_eager=True)

    tok = llm.get_tokenizer()

    sp = SamplingParams(temperature=0.0, max_tokens=3)

    def generate(prompts):
        reqs = [{"prompt_token_ids": tok(tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True), add_special_tokens=False).input_ids} for p in prompts]

        return [o.outputs[0].text for o in llm.generate(reqs, sp, use_tqdm=True)]

    return generate


def make_hf(model_name):
    import torch

    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_name, padding_side="left")

    model = AutoModelForCausalLM.from_pretrained(model_name).to(torch.float16).cuda().eval()

    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

    def generate(prompts):
        texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts]

        replies = []

        for i in range(0, len(texts), 16):
            batch = tok(texts[i:i + 16], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")

            with torch.no_grad():
                out = model.generate(**batch, max_new_tokens=3, do_sample=False, pad_token_id=pad)

            replies += tok.batch_decode(out[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)

        return replies

    return generate


def grades_of(replies):
    found = [re.findall(r"[123]", t) for t in replies]

    return [int(f[0]) if f else 0 for f in found]


def main():
    model_name, backend, src, dst = sys.argv[1:5]

    items = json.load(open(src))

    grades = []

    if items:
        generate = (make_vllm if backend == "vllm" else make_hf)(model_name)

        prompts = [JUDGE_PROMPT.format(q=q, r=r) for _, q, r in items]

        smoke = grades_of(generate(prompts[:8]))

        print("smoke test grades:", smoke, flush=True)

        if sum(g == 0 for g in smoke) > len(smoke) // 2:
            print("the judge does not answer with a digit, exiting with code 3", flush=True)

            sys.exit(3)

        grades = grades_of(generate(prompts))

    json.dump({key: g for (key, _, _), g in zip(items, grades)}, open(dst, "w"))


if __name__ == "__main__":
    main()

In [ ]:
# -- run the judge on both GPUs --

def judge_path(name):
    return os.path.join(WORK, "judge", f"{name}.json")

GRADES = {n: json.load(open(judge_path(n))) if os.path.exists(judge_path(n)) else {} for n in DONE}

items = []

for name in DONE:
    k = JUDGE_PER_PROMPT if name in MAIN else min(5, JUDGE_PER_PROMPT)

    for r in load_gen(name):
        for i, t in enumerate(r["responses"][:k]):
            key = f"{r['pid']}_{i}"

            if key in GRADES[name]:
                continue

            if t.strip():
                items.append((f"{name}|{key}", PROMPTS[r["pid"]], trunc(t, 300)))
            else:
                GRADES[name][key] = 1

JUDGE_USED = None

for jm in JUDGE_MODELS if items else []:
    srcs = [os.path.join(WORK, "judge", f"_in_{g}.json") for g in range(N_GPUS)]

    for g, src in enumerate(srcs):
        json.dump(items[g::N_GPUS], open(src, "w"))

    codes = run_on_gpus([[VENV_PY, os.path.join(WORK, "scripts", "judge.py"), jm, BACKEND, src, src.replace("_in_", "_out_")] for src in srcs], "judge")

    if all(c == 0 for c in codes):
        for src in srcs:
            for full_key, g in json.load(open(src.replace("_in_", "_out_"))).items():
                name, key = full_key.split("|")

                if g > 0:
                    GRADES[name][key] = g

        JUDGE_USED = jm

        break

    print(jm, "did not work as a judge (exit codes", codes, "), see logs/judge_*.log")

print("judge:", JUDGE_USED or ("cached grades" if not items else "none"))

for name in DONE:
    json.dump(GRADES[name], open(judge_path(name), "w"))

    g = np.array(list(GRADES[name].values()))

    if len(g):
        print(f"{name}: {len(g)} judged, {100 * np.mean(g >= 2):.0f}% valid, mean grade {g.mean():.2f}")

## 5. Metrics

For every model and prompt: the mean pairwise cosine similarity of its responses (the paper's intra-model measure), trigram overlap (mean pairwise Jaccard of word-trigram sets, first 64 words), and the share of judged-valid responses. Error bars in the figures are 95% confidence intervals over the 100 prompts.

In [ ]:
# -- metrics per model and prompt --

def intra(E, pids, first=None):
    vals = []

    for p in range(len(PROMPTS)):
        X = E[pids == p][:first]

        S = X @ X.T

        n = len(X)

        vals.append((S.sum() - np.trace(S)) / (n * (n - 1)) if n >= 2 else np.nan)

    return np.array(vals)


def cross(EA, pa, EB, pb):
    return np.array([(EA[pa == p] @ EB[pb == p].T).mean() for p in range(len(PROMPTS))])


def trigrams(t):
    w = re.findall(r"[a-z0-9']+", trunc(t).lower())

    return set(zip(w, w[1:], w[2:]))


def jaccard(texts, pids):
    T = np.array(texts, dtype=object)

    vals = []

    for p in range(len(PROMPTS)):
        sets = [trigrams(t) for t in T[pids == p]]

        sims = [len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:] if a or b]

        vals.append(np.mean(sims) if sims else np.nan)

    return np.array(vals)


def valid_mask(name, pids):
    ok = np.full(len(pids), np.nan)

    seen = {}

    for idx, p in enumerate(pids):
        key = f"{p}_{seen.get(p, 0)}"

        seen[p] = seen.get(p, 0) + 1

        if key in GRADES.get(name, {}):
            ok[idx] = GRADES[name][key] >= 2

    return ok


def ci(x):
    x = np.asarray(x, dtype=float)

    x = x[~np.isnan(x)]

    return x.mean(), 1.96 * x.std(ddof=1) / np.sqrt(len(x))

M = {}

for name in DONE:
    texts, pids = flat(name)

    T = np.array(texts, dtype=object)

    m = {"words": np.array([np.mean([words(t) for t in T[pids == p]]) for p in range(len(PROMPTS))])}

    for variant in VARIANTS:
        E = load_emb(name, variant)

        if E is not None:
            m[variant] = intra(E, pids)

    m["jaccard"] = jaccard(texts, pids)

    ok = valid_mask(name, pids)

    if not np.isnan(ok).all():
        m["valid"] = np.array([np.nanmean(ok[pids == p]) for p in range(len(PROMPTS))])

        E = load_emb(name, "main_trunc")

        if E is not None:
            m["sim_valid"] = intra(E[ok == 1], pids[ok == 1])

    M[name] = m

NAMES = {"words": "words", "main_full": "sim Qwen3-Emb full", "main_trunc": "sim Qwen3-Emb 64w", "second_trunc": "sim mpnet 64w", "jaccard": "trigram Jaccard", "valid": "valid share", "sim_valid": "sim valid only"}

SUMMARY = pd.DataFrame({JOB[n]["label"]: {NAMES[k]: np.nanmean(v) for k, v in M[n].items()} for n in M}).T.round(3)

SUMMARY.to_csv(os.path.join(WORK, "tables", "summary.csv"))

print(SUMMARY.to_string())

## 6. Replication

The paper's Table 6 gives, for each model, the share of prompts whose average pairwise similarity falls in each 0.1-wide range. For Qwen2.5-1.5B-Instruct most prompts land between 0.6 and 0.9.

Different embedding models put similarity on different scales, so the comparison uses a random baseline: the similarity of two responses to different prompts. The paper reports 0.1-0.2 for random pairs; this notebook measures its own. Normalized similarity = (similarity - random) / (1 - random): 0 means as similar as answers to unrelated prompts, 1 means identical. If the normalized means match, the replication holds even though the embedding model differs.

In [ ]:
# -- plot style: validated colors for the three families, text in ink colors --

import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({"font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#0b0b0b", "text.color": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True, "figure.facecolor": "white", "savefig.dpi": 200, "savefig.bbox": "tight"})

COLOR = {"OLMo 2": "#2a78d6", "Qwen2.5": "#eb6834", "SmolLM2": "#1baf7a"}

MARKER = {"OLMo 2": "o", "Qwen2.5": "s", "SmolLM2": "^"}

GRAY = "#898781"

LIGHT, DARK = "#86b6ef", "#1c5cab"

BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])


def save(fig, name):
    fig.savefig(os.path.join(WORK, "figures", name))

    plt.show()

In [ ]:
# -- replication: per-prompt similarity histogram vs the paper (Table 6) --

PAPER_QWEN = np.array([0, 0, 1, 2, 8, 11, 16, 20, 39, 3])

PAPER_RANDOM = 0.15

BINS = [f"{a / 10:.1f}-{(a + 1) / 10:.1f}" for a in range(10)]


def hist(sims):
    counts = np.histogram(np.clip(sims[~np.isnan(sims)], 0, 0.9999), bins=np.linspace(0, 1, 11))[0]

    return 100 * counts / counts.sum()


def normalized(s, r):
    return (s - r) / (1 - r)


def random_baseline(variant, n_pairs=20000):
    names = [n for n in MAIN if os.path.exists(emb_path(n, variant))]

    E = np.concatenate([load_emb(n, variant) for n in names])

    P = np.concatenate([flat(n)[1] for n in names])

    rng = np.random.default_rng(0)

    a, b = rng.integers(0, len(E), n_pairs), rng.integers(0, len(E), n_pairs)

    keep = P[a] != P[b]

    return float((E[a[keep]] * E[b[keep]]).sum(1).mean())

if "main_full" in M.get("qwen_inst", {}):
    RANDOM = random_baseline("main_full")

    ours = float(np.nanmean(M["qwen_inst"]["main_full"]))

    paper = float((np.arange(10) / 10 + 0.05) @ PAPER_QWEN / 100)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)

    for ax, values, color, r, title in [(axes[0], PAPER_QWEN, GRAY, PAPER_RANDOM, f"Paper (text-embedding-3-small)\nmean {paper:.2f}, normalized {normalized(paper, PAPER_RANDOM):.2f}"), (axes[1], hist(M["qwen_inst"]["main_full"]), COLOR["Qwen2.5"], RANDOM, f"This notebook (Qwen3-Embedding-0.6B)\nmean {ours:.2f}, normalized {normalized(ours, RANDOM):.2f}")]:
        ax.bar(np.arange(10), values, width=0.8, color=color)

        ax.axvline(10 * r - 0.5, color="#52514e", linewidth=1.5)

        ax.text(10 * r - 0.4, 0.93, "random pairs", color="#52514e", fontsize=11, transform=ax.get_xaxis_transform())

        ax.set_xticks(np.arange(10), BINS, rotation=30, ha="right")

        ax.set_xlabel("Average pairwise similarity of the 50 responses to a prompt")

        ax.set_title(title)

    axes[0].set_ylabel("Share of prompts (%)")

    fig.suptitle("Qwen2.5-1.5B-Instruct: paper vs replication", fontsize=14)

    fig.tight_layout()

    save(fig, "fig1_replication.png")

    print(f"random-pair similarity here: {RANDOM:.3f} (paper: 0.1-0.2)")

HIST = pd.DataFrame({JOB[n]["label"]: hist(M[n]["main_full"]) for n in MAIN if "main_full" in M.get(n, {})}, index=BINS).T.round(0)

HIST.to_csv(os.path.join(WORK, "tables", "histogram_table6_style.csv"))

print(HIST.to_string())

## 7. Where does the hivemind come from?

Left to right: the same OLMo 2 1B model after pretraining (base), supervised fine-tuning, DPO, and RLVR (final). Qwen2.5 and SmolLM2 only release a base and a final instruct model, so they appear at the two ends (dashed lines). Six views of the same question:
- the main measure (Qwen3-Embedding, full responses)
- the same on the first 64 words only (length control)
- a different embedding model (all-mpnet-base-v2)
- word overlap instead of embeddings (trigram Jaccard)
- similarity among judged-valid responses only
- the share of judged-valid responses (quality check)

If similarity rises from base to post-trained models in every panel, the homogeneity comes from post-training, not from the metric, from length, or from off-task text. The table after the figure gives each step's change with its 95% confidence interval.

In [ ]:
# -- figure: similarity by training stage --

PANELS = [("main_full", "Qwen3-Embedding, full response"), ("main_trunc", "Qwen3-Embedding, first 64 words"), ("second_trunc", "mpnet embeddings, first 64 words"), ("jaccard", "Trigram overlap, first 64 words"), ("sim_valid", "Valid responses only, first 64 words"), ("valid", "Share of judged-valid responses")]

PANELS = [(k, t) for k, t in PANELS if any(k in M.get(n, {}) for n in MAIN)]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (key, title) in zip(axes.flat, PANELS):
    for fam in FAMILIES:
        pts = [(STAGES.index(JOB[n]["stage"]),) + ci(M[n][key]) for n in MAIN if JOB[n]["family"] == fam and key in M.get(n, {})]

        if pts:
            xs, ys, es = zip(*pts)

            ax.errorbar(xs, ys, yerr=es, color=COLOR[fam], marker=MARKER[fam], markersize=8, linewidth=2, capsize=4, linestyle="-" if len(pts) > 2 else "--", label=fam)

    ax.set_title(title)

    ax.set_xticks(range(len(STAGES)), STAGES)

    ax.set_xlim(-0.3, len(STAGES) - 0.7)

for ax in axes.flat[len(PANELS):]:
    ax.axis("off")

handles, labels = max((a.get_legend_handles_labels() for a in axes.flat), key=lambda h: len(h[1]))

fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)

fig.suptitle("Similarity of responses to the same prompt, by training stage (higher = more homogeneous)", fontsize=14)

fig.tight_layout(rect=(0, 0.05, 1, 0.97))

save(fig, "fig2_stages.png")

In [ ]:
# -- paired differences between stages (mean over prompts, 95% CI) --

COMPARE = [("OLMo 2: SFT - base", "olmo_base", "olmo_sft"), ("OLMo 2: DPO - SFT", "olmo_sft", "olmo_dpo"), ("OLMo 2: final - DPO", "olmo_dpo", "olmo_final"), ("OLMo 2: final - base", "olmo_base", "olmo_final"), ("Qwen2.5: inst - base", "qwen_base", "qwen_inst"), ("SmolLM2: inst - base", "smol_base", "smol_inst")]

METRICS = ["main_full", "main_trunc", "second_trunc", "jaccard", "sim_valid", "valid"]

DIFFS = pd.DataFrame(index=[c[0] for c in COMPARE], columns=[NAMES[k] for k in METRICS], dtype=object)

for label, a, b in COMPARE:
    for key in METRICS:
        if key in M.get(a, {}) and key in M.get(b, {}):
            mean, half = ci(M[b][key] - M[a][key])

            DIFFS.loc[label, NAMES[key]] = f"{mean:+.3f} ± {half:.3f}"

DIFFS = DIFFS.dropna(how="all", axis=1).fillna("")

DIFFS.to_csv(os.path.join(WORK, "tables", "stage_differences.csv"))

print(DIFFS.to_string())

## 8. Across families

The paper's strongest claim is that different models produce strikingly similar outputs. Here each pair of families is compared twice: their base models, and their post-trained models. Raw similarity mixes two effects, each model being narrow and models converging on each other, so the right panel divides cross-model similarity by the geometric mean of the two models' own similarity. A value near 1 means the two models' outputs are as similar to each other as each model's outputs are among themselves.

In [ ]:
# -- cross-model similarity heatmap (first 64 words) --

AV = [n for n in MAIN if os.path.exists(emb_path(n, "main_trunc"))]

EMB = {n: (load_emb(n, "main_trunc"), flat(n)[1]) for n in AV}

C = np.array([[np.nanmean(intra(*EMB[a]) if a == b else cross(*EMB[a], *EMB[b])) for b in AV] for a in AV])

if len(AV) > 1:
    fig, ax = plt.subplots(figsize=(9.5, 7.5))

    im = ax.imshow(C, cmap=BLUES, vmin=C.min(), vmax=C.max())

    for i in range(len(AV)):
        for k in range(len(AV)):
            ax.text(k, i, f"{100 * C[i, k]:.1f}", ha="center", va="center", fontsize=10, color="white" if C[i, k] > C.min() + 0.55 * (C.max() - C.min()) else "#0b0b0b")

    ax.set_xticks(range(len(AV)), [JOB[n]["label"] for n in AV], rotation=35, ha="right")

    ax.set_yticks(range(len(AV)), [JOB[n]["label"] for n in AV])

    ax.grid(False)

    ax.set_title("Similarity between models' responses to the same prompt (x100)\ndiagonal: within one model")

    fig.colorbar(im, ax=ax, shrink=0.8, label="mean cosine similarity")

    save(fig, "fig3_heatmap.png")

In [ ]:
# -- cross-family: base pairs vs post-trained pairs --

BASE = {"OLMo 2": "olmo_base", "Qwen2.5": "qwen_base", "SmolLM2": "smol_base"}

FINAL = {"OLMo 2": "olmo_final", "Qwen2.5": "qwen_inst", "SmolLM2": "smol_inst"}

POS = {n: i for i, n in enumerate(AV)}

rows = []

for fa, fb in [("OLMo 2", "Qwen2.5"), ("OLMo 2", "SmolLM2"), ("Qwen2.5", "SmolLM2")]:
    for stage, pick in [("base", BASE), ("post-trained", FINAL)]:
        a, b = POS.get(pick[fa]), POS.get(pick[fb])

        if a is not None and b is not None:
            rows.append(dict(pair=f"{fa} - {fb}", stage=stage, raw=C[a, b], normalized=C[a, b] / np.sqrt(C[a, a] * C[b, b])))

XFAM = pd.DataFrame(rows, columns=["pair", "stage", "raw", "normalized"])

XFAM.to_csv(os.path.join(WORK, "tables", "cross_family.csv"), index=False)

print(XFAM.round(3).to_string(index=False))

if len(XFAM):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    pairs = list(dict.fromkeys(XFAM["pair"]))

    for ax, col, title in [(axes[0], "raw", "Similarity between two families"), (axes[1], "normalized", "Relative to each model's own similarity")]:
        tab = XFAM.pivot(index="pair", columns="stage", values=col).reindex(pairs)

        for i, p in enumerate(pairs):
            ax.plot([i, i], [tab.loc[p].min(), tab.loc[p].max()], color="#c3c2b7", linewidth=2, zorder=1)

        for stage, color in [("base", LIGHT), ("post-trained", DARK)]:
            if stage in tab:
                ax.scatter(range(len(pairs)), tab[stage], s=110, color=color, edgecolor="white", linewidth=1, zorder=2, label=stage)

        ax.set_xticks(range(len(pairs)), [p.replace(" - ", " vs\n") for p in pairs])

        ax.set_xlim(-0.5, len(pairs) - 0.5)

        ax.set_title(title)

    fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False)

    fig.tight_layout(rect=(0, 0.08, 1, 1))

    save(fig, "fig4_cross_family.png")

## 9. "Write a metaphor involving time."

The paper's Figure 1: 50 responses from each of 25 models fall into two clusters, "time is a river" and "time is a weaver". Here all responses to the same prompt are projected with one PCA, and each panel highlights one model. Panel titles give the share of that model's responses mentioning a river or a weaver.

In [ ]:
# -- metaphor prompt: one PCA over all models' responses --

from sklearn.decomposition import PCA

AVF = [n for n in MAIN if os.path.exists(emb_path(n, "main_full"))]

PARTS = {n: load_emb(n, "main_full")[flat(n)[1] == METAPHOR] for n in AVF}

pca = PCA(2).fit(np.concatenate(list(PARTS.values())))

PROJ = {n: pca.transform(E) for n, E in PARTS.items()}

ALL = np.concatenate(list(PROJ.values()))

fig, axes = plt.subplots(2, 4, figsize=(16, 8.4), sharex=True, sharey=True)

for ax, n in zip(axes.flat, AVF):
    texts = load_gen(n)[METAPHOR]["responses"]

    river = 100 * np.mean(["river" in t.lower() for t in texts])

    weaver = 100 * np.mean(["weav" in t.lower() for t in texts])

    ax.scatter(ALL[:, 0], ALL[:, 1], s=12, color="#c3c2b7", alpha=0.6)

    ax.scatter(PROJ[n][:, 0], PROJ[n][:, 1], s=36, color=COLOR[JOB[n]["family"]], edgecolor="white", linewidth=0.8)

    ax.set_title(f"{JOB[n]['label']}\nriver {river:.0f}%, weaver {weaver:.0f}%")

    ax.set_xticks([])

    ax.set_yticks([])

for ax in axes.flat[len(AVF):]:
    ax.axis("off")

fig.suptitle('Responses to "Write a metaphor involving time." (one PCA for all models)', fontsize=14)

fig.tight_layout()

save(fig, "fig5_metaphor_pca.png")

with open(os.path.join(WORK, "tables", "metaphor_examples.txt"), "w") as f:
    for n in AVF:
        f.write(f"== {JOB[n]['label']} ==\n" + "\n".join("- " + t.replace("\n", " ")[:300] for t in load_gen(n)[METAPHOR]["responses"][:10]) + "\n\n")

## 10. Can decoding or prompting undo it?

Three cheap fixes on the final OLMo model, 20 responses per prompt each: a higher temperature (1.5), min-p sampling at temperature 2.0 (the paper's diversity setting), and list prompting, which asks for 5 different answers in one call (a simplified form of verbalized sampling). The references are the base model and the final model at temperature 1.0, using the first 20 of their 50 responses. A fix only counts if it lowers similarity without losing valid responses.

In [ ]:
# -- mitigations on the final OLMo model --

SETTINGS = [("olmo_base", "Base\nT=1.0"), ("olmo_final", "Final\nT=1.0"), ("olmo_final_t15", "Final\nT=1.5"), ("olmo_final_minp", "Final\nmin-p"), ("olmo_final_list", "Final\nlist x5")]

rows = []

for name, label in SETTINGS:
    if not os.path.exists(emb_path(name, "main_trunc")):
        continue

    pids = flat(name)[1]

    sim, half = ci(intra(load_emb(name, "main_trunc"), pids, first=20))

    ok = valid_mask(name, pids)

    first5 = np.concatenate([np.where(pids == p)[0][:5] for p in range(len(PROMPTS))])

    rows.append(dict(setting=label.replace("\n", " "), label=label, sim=sim, sim_ci=half, valid=np.nanmean(ok[first5])))

MIT = pd.DataFrame(rows, columns=["setting", "label", "sim", "sim_ci", "valid"])

MIT.drop(columns="label").to_csv(os.path.join(WORK, "tables", "mitigations.csv"), index=False)

print(MIT.drop(columns="label").round(3).to_string(index=False))

if len(MIT):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    x = np.arange(len(MIT))

    colors = [GRAY if s.startswith("Base") else COLOR["OLMo 2"] for s in MIT["setting"]]

    axes[0].bar(x, MIT["sim"], yerr=MIT["sim_ci"], color=colors, width=0.6, capsize=4, ecolor="#52514e")

    axes[0].set_title("Similarity, first 64 words (lower = more diverse)")

    axes[1].bar(x, 100 * MIT["valid"], color=colors, width=0.6)

    axes[1].set_title("Judged valid responses (%)")

    for ax in axes:
        ax.set_xticks(x, MIT["label"])

    save(fig, "fig6_mitigations.png")

## 11. Numbers for the slides

Everything the figures show, in one place. To download from /kaggle/working/hivemind: figures/*.png, tables/*.csv, tables/metaphor_examples.txt, tables/run_info.json (versions and settings, for the tools-and-workflow slide), and gen/*.jsonl with every response.

In [ ]:
# -- summary --

print(SUMMARY.to_string(), "\n")

print(DIFFS.to_string(), "\n")

print(XFAM.round(3).to_string(index=False), "\n")

print(MIT.drop(columns="label").round(3).to_string(index=False), "\n")

INFO = dict(date=time.strftime("%Y-%m-%d"), vllm=VLLM_VERSION, backend=BACKEND, gpus=gpu_list, judge=JUDGE_USED or JUDGE_MODELS, embeddings=[EMB_MAIN, EMB_SECOND], sampling=dict(n=N_SAMPLES, top_p=TOP_P, temperature=TEMPERATURE, chat_max_tokens=CHAT_MAX_TOKENS, base_max_tokens=BASE_MAX_TOKENS), jobs=CHECK[["dtype", "backend"]].to_dict("index"))

json.dump(INFO, open(os.path.join(WORK, "tables", "run_info.json"), "w"), indent=1)

print("files:", *sorted(os.path.relpath(p, WORK) for p in glob.glob(os.path.join(WORK, "figures", "*")) + glob.glob(os.path.join(WORK, "tables", "*"))), sep="\n  ")